# Train Your First E2E Wake Word Model with Nanowakeword

Welcome to the **E2E (End-to-End) training** tutorial for Nanowakeword!

In this notebook, we will train a wake word model that operates directly on **raw audio waveforms** instead of pre-computed embeddings. This eliminates the need for feature extraction and allows the model to learn optimal representations directly from the audio signal.

**E2E model types available:**
- `e2e_dnn` - Lightweight DNN with a raw audio frontend (fastest, smallest)
- `e2e_cnn` - CNN with a raw audio frontend (moderate)
- `e2e_quartznet` - QuartzNet-style architecture with configurable blocks (most expressive)

**Our goal:** Go from zero to a ready-to-use E2E wake word model in just a few simple steps. Let's get started!

**Installation**

In [ ]:
# @title Step 1: Install Nanowakeword
# We install the full [train] package to get all the necessary dependencies.

! pip install "nanowakeword[train] @ git+https://github.com/arcosoph/nanowakeword.git"  # Or download from pypi
! pip install piper-tts

print("Installation complete!")

## Step 2: Prepare the Dataset

A great model starts with great data. For this tutorial, we will:
1.  **Download** the SonicWeave-v2 background noise dataset (extracts to `SonicWeave-v2/part1`, `part2`, `part3`).
2.  **Download** Room Impulse Response (RIR) data.
3.  **Organize** all your project files within a clean, well-structured folder hierarchy.

In [ ]:
# @title Step 2.1: Download Background Noise Dataset (SonicWeave-v2)
# SonicWeave-v2 contains diverse background audio clips for training augmentation.
# The zip extracts into SonicWeave-v2/part1, SonicWeave-v2/part2, SonicWeave-v2/part3

! wget -q -O sonicweave_v2.zip "https://huggingface.co/datasets/arcosoph/datasets_zip/resolve/main/audio/SonicWeave-v2.zip"
! unzip -q sonicweave_v2.zip && rm sonicweave_v2.zip

print("Background noise dataset downloaded and extracted.")

### Download RIR Data (Optional)

Room Impulse Responses add reverberation, making your model more robust to different acoustic environments.

In [ ]:
! mkdir -p data/rir
! wget -q -O data/rir/rir_0.5s_16k.h5 https://github.com/SeanNiemar/rir-generation-toolkit/releases/download/v2.2/rir_0.5s_16k.h5
print("RIR data downloaded.")

## Step 3: Configure and Train the E2E Model

Now for the fun part! We will create a `config.yaml` file specifically for **E2E training**. The key differences from embedding mode are:

- `mode: "e2e"` - trains directly on raw audio waveforms
- `model_type` must be one of `e2e_dnn`, `e2e_cnn`, or `e2e_quartznet`
- `clip_samples` - the fixed number of audio samples per clip (e.g. 16000 = 1 second at 16 kHz)
- `data_manifest` points to directories of WAV files (not `.npy` feature files)
- `data_generation_manifest` is used instead of `feature_generation_manifest`

Full config explanation: https://arcosoph.com/blog/nanowakeword_config_guide

In [ ]:
# @title Step 3.1: Create the E2E Configuration File

import yaml

config_dict = {
    # ==========================================================
    # Project & Data Paths
    # ==========================================================
    "model_name": "arcosoph_e2e_A_v1",  # Change this name when creating a new model
    "output_dir": "./trained_models",

    "positive_data_path": "./data/positive",
    "negative_data_path": "./data/negative",
    "background_paths": ["./SonicWeave-v2/part1", "./SonicWeave-v2/part2", "./SonicWeave-v2/part3"],  # Background noise for augmentation
    "rir_paths": ["./data/rir"],  # Optional: leave empty if you do not want RIR augmentation

    # ==========================================================
    # Pipeline Control
    # ==========================================================
    "mode": "e2e",  # End-to-end training on raw waveforms

    "generate_clips": True,       # Generates synthetic TTS audio clips
    "transform_clips": True,       # Augments and prepares clips for training
    "train_model": True,           # Runs the training loop
    "overwrite": False,            # Set True to skip existing files

    # ==========================================================
    # E2E Model Architecture
    # ==========================================================
    "model_type": "e2e_dnn",  # Choose: e2e_dnn, e2e_cnn, or e2e_quartznet
    "e2e_frontend_channels": 32,  # Number of channels in the raw audio frontend Conv1d
    "e2e_frontend_depth": 2,     # Number of layers in the frontend Conv1d stack
    # "e2e_quartznet_config": [[64, 11, 1], [64, 13, 1], [64, 17, 1]],  # Only used for e2e_quartznet

    "embedding_dim": 64,         # Output embedding dimension
    "dropout_prob": 0.3,
    "activation_function": "relu",  # relu or silu

    # ==========================================================
    # Training Settings
    # ==========================================================
    "steps": 50000,            # Total training steps
    "stabilization_steps": 40000,  # Steps before validation/early stopping activates

    "clip_samples": 16000,     # Fixed clip length in samples (1 second at 16 kHz)
    "sample_rate": 16000,      # Audio sample rate in Hz
    "num_workers": 2,          # Number of DataLoader worker processes

    "optimizer_type": "adamw",
    "learning_rate_max": 0.0005,
    "lr_scheduler_type": "onecycle",
    "weight_decay": 0.01,
    "momentum": 0.9,

    # ==========================================================
    # Loss Function
    # ==========================================================
    "margin_pos": 2.0,
    "margin_neg": -2.0,
    "LOSS_BIAS": 0.65,

    "logit_reg_weight": 0.0005,
    "logit_reg_margin": 4.0,
    "logit_min_margin": 1.5,

    # ==========================================================
    # Batch Composition
    # ==========================================================
    "batch_composition": {
        "t": 100,
        "n": 100,
        "b": 90,
        "hn": 20,
        "no1": 30,
        "no2": 30,
        "no3": 30,
    },

    # ==========================================================
    # Synthetic Data Generation (TTS)
    # ==========================================================
    "target_phrase": "hello arcosoph",  # Your wake word

    "data_generation_tasks": [
        {
            "name": "positive_samples",
            "enabled": True,
            "output_dir": "data/positive",
            "num_samples": 2500,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "hello arcosoph"
            }
        },
        {
            "name": "positive_val_samples",
            "enabled": True,
            "output_dir": "data/positive_val",
            "num_samples": 2000,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "hello arcosoph"
            }
        },
        {
            "name": "adversarial_negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 5000,
            "file_prefix": "neg_auto",
            "text_source": {
                "type": "auto_adversarial",
                "base_phrase": "hello arcosoph",
                "include_input_words": True,
                "include_partial_phrase": True,
                "multi_word_prob": 0.5,
                "max_multi_word_len": 3
            }
        },
        {
            "name": "phoneme_hard_negatives",
            "enabled": True,
            "output_dir": "data/negative_phoneme",
            "num_samples": 3000,
            "file_prefix": "neg_ph",
            "text_source": {
                "type": "phoneme_adversarial",
                "base_phrase": "hello arcosoph",
                "min_distance": 0.3
            }
        },
        {
            "name": "custom_negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 50,
            "file_prefix": "neg_custom",
            "text_source": {
                "type": "from_list",
                "phrases": [
                    "arcosoph eloo",
                    "arcosop",
                    "hie arcosoph",
                    "arkhoshap",
                    "yarkasop"
                ],
                "repeat_each": 10
            }
        }
    ],

    # ==========================================================
    # Augmentation
    # ==========================================================
    "augmentation_batch_size": 16,
    "feature_gen_cpu_ratio": 1.0,

    "augmentation_settings": {
        "gain_prob": 1.0,
        "max_gain_in_db": 2.0,
        "max_pitch_semitones": 1.0,
        "max_snr_in_db": 35.0,
        "min_gain_in_db": -2.0,
        "min_pitch_semitones": -1.0,
        "min_snr_in_db": 15.0,
        "pitch_prob": 0.3,
        "rir_prob": 0.0
    },

    # ==========================================================
    # Audio Generation Manifest
    # ==========================================================
    # In E2E mode, this manifest generates augmented WAV clips that
    # are then used directly for training (no feature extraction needed).
    "data_generation_manifest": {
        "pos_feature": {
            "input_audio_dirs": ["./data/positive"],
            "output_dir": "./data/positive",
            "file_prefix": "pos_aug",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10,
            "num_samples": 25000
        },
        "pos_val_feature": {
            "input_audio_dirs": ["./data/positive_val"],
            "output_dir": "./data/positive_val",
            "file_prefix": "pos_val_aug",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10,
            "num_samples": 20000
        },
        "neg_feature": {
            "input_audio_dirs": ["./data/negative"],
            "output_dir": "./data/negative",
            "file_prefix": "neg_aug",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10,
            "num_samples": 50000
        },
        "neg_ph_feature": {
            "input_audio_dirs": ["./data/negative_phoneme"],
            "output_dir": "./data/negative_phoneme",
            "file_prefix": "neg_ph_aug",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 1,
            "num_samples": 3000
        }
    },

    "background_paths_duplication_rate": [1],
    # ==========================================================
    # Data Manifest (Directories of WAV files for training)
    # ==========================================================
    # In E2E mode, each key maps to a directory path containing .wav files.
    # The number after each path is the sampling weight.
    "data_manifest": {
        "targets": {
            "t": "./data/positive_aug"
        },
        "negatives": {
            "n": "./data/negative_aug",
            "hn": "./data/negative_phoneme_aug",
            "no1": "./SonicWeave-v2/part1",
            "no2": "./SonicWeave-v2/part2",
            "no3": "./SonicWeave-v2/part3"
        },
        "targets_val": {
            "t_v": "./data/positive_val_aug"
        }
    },

    # ==========================================================
    # Validation
    # ==========================================================
    "val_miss_weight": 4.0,
    "val_fp_weight": 1.0,
    "validation_batch_size": 256,
    "validation_smoothing_window": 3,
    "val_early_stopping_patience": 6000,

    # ==========================================================
    # Curriculum Learning (ISBL Sampling)
    # ==========================================================
    "hardness_ema_alpha": 0.05,
    "hardness_floor": 0.05,
    "hardness_reset_interval": 5000,
    "hardness_reset_decay": 0.5,
    "checkpoint_averaging_top_k": 5,

    # ==========================================================
    # Checkpointing & Debug
    # ==========================================================
    "checkpointing": {
        "enabled": True,
        "interval_steps": 1000,
        "limit": 2
    },

    "early_stopping_patience": 0,
    "min_delta": 0.0001,
    "ema_alpha": 0.01,

    "onnx_opset_version": 17,
    "show_training_summary": True,
    "debug_mode": True

}

config_path = "./config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print("config.yaml written successfully")
print("NOTE: After the first run, set transform_clips: False and generate_clips: False")
print("      to avoid re-generating clips. Only re-enable if you need to regenerate data.")


**Run Training!**

In [ ]:
# @title Step 3.2: Run the Magic Command!
# This command will do everything: generate TTS clips, augment data, and train the E2E model.
# It might take some time depending on the hardware (especially on a CPU).

from nanowakeword.trainer import train

args_list = [
    '--config_path', f'{config_path}',
]

print("Starting E2E NanoWakeWord training...")

try:
    train(args_list)
    print("\n\nCONGRATULATIONS!")
    print("Your custom E2E wake word model has been successfully trained!")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

## What's Next?

You have successfully trained your own custom E2E wake word model!

You can now download the `.onnx` or `.pt` file from the `trained_models` directory (check the file browser on the left) and use it in your own applications.

For more advanced topics, such as using your own datasets or fine-tuning the configuration, please check out our full documentation on **[GitHub](https://github.com/arcosoph/nanowakeword)**.

---
## Step 4: Save Your Model to Google Drive

The final step is to save your trained model and performance graph to a safe and accessible place. Instead of a slow direct download, we will save the files directly to your Google Drive. This process is almost instantaneous.

Run the cells below to:
1.  Connect your Google Drive account.
2.  Copy all the trained files into a new folder named `nanowakeword_models` in your Drive.

In [ ]:
# @title Step 4.1: Connect to Google Drive
# This will ask for your permission to access your Google Drive.

from google.colab import drive

try:
    drive.mount('/content/drive')
    print("\nGoogle Drive connected successfully!")
except Exception as e:
    print(f"An error occurred while connecting to Google Drive: {e}")

In [ ]:
# @title Step 4.2: Copy Final Model and Artifacts to Google Drive

import os
import shutil

# --- Configuration ---
# Get model_name and output_dir from the config_dict defined earlier
model_name = config_dict.get("model_name", "my_model")
output_dir = config_dict.get("output_dir", "./trained_models")

# --- Source and Destination Paths ---
source_project_dir = os.path.join(output_dir, model_name)
drive_destination_dir = f"drive/MyDrive/nanowakeword_models/{model_name}"

# --- Start Copy Process ---
print("Starting the process to copy trained files to Google Drive...")

if not os.path.exists(source_project_dir):
    print(f"\nERROR: Source directory not found at '{source_project_dir}'")
    print("This indicates that the training process did not create the expected output folder.")
    print("Please ensure the training step completed successfully before running this cell.")
else:
    if os.path.exists(drive_destination_dir):
        print(f"Found an existing folder in Drive. Removing it for a fresh copy: '{drive_destination_dir}'")
        shutil.rmtree(drive_destination_dir)

    try:
        shutil.copytree(source_project_dir, drive_destination_dir)

        print("=" * 50)
        print("SUCCESS! All files have been saved to your Google Drive.")
        print("=" * 50)
        print(f"\nYour complete project, including the model and performance graphs, can be found in:")
        print(f"  '{drive_destination_dir}'")

        print("\nContents of the saved folder:")
        for root, dirs, files in os.walk(drive_destination_dir):
            level = root.replace(drive_destination_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = ' ' * 4 * (level + 1)
            for f in files:
                print(f"{sub_indent}{f}")

    except Exception as e:
        print(f"\nERROR: An unexpected error occurred during the copy process.")
        print(f"Details: {e}")

<p style="font-size:18px; font-weight:600;">
  If you find this helpful, please support us at
  <a href="https://arcosoph.com" style="text-decoration:none;">
    <span style="color:#fefefe;">A</span>
    <span style="color:#2cab4e;">r</span>
    <span style="color:#029adb;">c</span>
    <span style="color:#821720;">o</span>
    <span style="color:#f9e91b;">s</span>
    <span style="color:#821720;">o</span>
    <span style="color:#fefefe;">p</span>
    <span style="color:#f9e91b;">h</span>
  </a> or give our
  <a href="https://github.com/arcosoph/nanowakeword" style="color:#007BFF; font-weight:bold; text-decoration:none;">
    repository
  </a> a star.
</p>